# Evo Kriging Task

The purpose of this notebook is to test the Evo Kriging task by performing an estimation and ultimately saving the results to blocksync such that they can be viewed in leapfrog.

---

First import necessary packages

In [3]:
import pyarrow as pa
from dotenv import dotenv_values
from evo.blockmodels import BlockModelAPIClient
from evo.blockmodels.endpoints.models import ColumnHeaderType
from evo.widgets import FeedbackWidget, ServiceManagerWidget
from evo.objects import ObjectAPIClient

from evo.compute import JobClient

## Authentication

Next login to Seequent Evo and select an instance and workspace

In [4]:
# autheticate and create service manager

evo_config = dotenv_values(".evo.env")
manager = await ServiceManagerWidget.with_auth_code(**evo_config).login()

/home/eric/code-dev/evo-python-sdk/packages/evo-sdk-common/src/evo/notebooks/authorizer.py:108: UserWarning: The evo.notebooks.AuthorizationCodeAuthorizer is not secure, and should only ever be used in Jupyter notebooks in a private environment.
  warnings.warn(


ServiceManagerWidget(children=(VBox(children=(HBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR…

## Establish Object API client
Now, establish a connection to the geoscience objects API

In [5]:
# connect to Object API
environment = manager.get_environment()
connector = manager.get_connector()

object_client = ObjectAPIClient(environment, connector)
service_health = await object_client.get_service_health()

print(f"Object API is {service_health.status.name.lower()}")

data_client = object_client.get_data_client(manager.cache)

Object API is healthy


### Select Objects for Kriging
First, list all of the objects present in the workspace such that the desired objects can be selected as input for kriging

In [6]:
# list objects in the workspace

offset = 0
while True:
    page = await object_client.list_objects(offset=offset, limit=10)
    if offset == 0:
        print(f"Found {page.total} object{'' if page.total == 1 else 's'}")
    for object in page:
        print(f"{object.path}: <{object.schema_id}> ({object.id})")

    if page.is_last:
        break
    else:
        offset = page.next_offset

Found 46 objects
/Au_ppm in Smokey - INTR1: Transformed Variogram Model.json: </objects/variogram/1.1.0/variogram.schema.json> (0c89856c-4a82-41c0-9b34-77f9ff382d4b)
/Au_ppm in Smokey - INTR2: Transformed Variogram Model.json: </objects/variogram/1.1.0/variogram.schema.json> (39e97136-ab5b-4946-a3bc-176b55344d15)
/Au_ppm in Smokey - INTR3: Transformed Variogram Model.json: </objects/variogram/1.1.0/variogram.schema.json> (e4bb722b-039d-4a4e-9718-1fd82799ecdd)
/con_sim_objects/277cf76c-9cfa-4192-a0c9-8441baf6e162/Au_ppm Values.json: </objects/pointset/1.2.0/pointset.schema.json> (816c7c70-cdc2-4109-bb5c-aebe0648de02)
/con_sim_objects/277cf76c-9cfa-4192-a0c9-8441baf6e162/Transformed Variogram Model.json: </objects/variogram/1.1.0/variogram.schema.json> (14c92b6b-9f7a-4b56-8740-28bee98ffe57)
/con_sim_objects/487ce545-0e42-4417-8627-0a925398dcdb/AU_PPM Values.json: </objects/pointset/1.2.0/pointset.schema.json> (3f89f789-7e59-4be6-b132-47706b927a7e)
/con_sim_objects/487ce545-0e42-4417-8627

 ### Download the desired objects

 Select and download the objects needed for kriging

In [7]:
# download needed objects

comps = await object_client.download_object_by_path("Smokey_Resource: comps_10m.json")
variogram_model = await object_client.download_object_by_path(
    "Au_ppm in Smokey - INTR2: Transformed Variogram Model.json"
)
block_model = await object_client.download_object_by_path("Resource Model.json")
mgrid = await object_client.download_object_by_path("INTR2-testevokriging.json")

## Perform the Kriging

Setup the kriging parameters and start the kriging

In [8]:
from evo.compute.endpoints.api import TasksApi

In [9]:
tasks_api = TasksApi(connector)

In [ ]:
# setup kriging parameters

kriging = await JobClient.submit(
    connector=connector,
    org_id=environment.org_id,
    topic="geostatistics",
    task="kriging",
    parameters={
        "source": {
            "object": comps.metadata.url,
            "attribute": "attributes[?name=='Au_ppm']",
        },
        "target": {
            "object": mgrid.metadata.url,
            "attribute": {"operation": "create", "name": "Au_ok-test-eric1"},
        },
        "kriging_method": {"type": "ordinary"},
        "variogram": variogram_model.metadata.url,
        "neighborhood": {
            "ellipsoid": {
                "ellipsoid_ranges": {"major": 50.0, "semi_major": 50.0, "minor": 50.0},
                "rotation": {"dip_azimuth": 0.0, "dip": 0.0, "pitch": 0.0},
            },
            "max_samples": 40,
        },
    },
    result_type=dict,
    preview=True,
)


print(kriging)

https://350mt.api.integration.seequent.com/compute/orgs/829e6621-0ab6-4d7d-96bb-2bb5b407a5fe/geostatistics/kriging/5611c752-b7de-4851-890f-7279da07e462/status


Check the status of the kriging until finished

In [32]:
# check the progress - check periodically until finished
response = await kriging.get_status()
print(response)

[in progress] 25% > Kriging method selected


In [35]:
# check the progress - check periodically until finished
response = await kriging.get_status()
print(response)

[in progress] 25% > Kriging method selected


Get the kriging results

In [27]:
# get the results when finished
await kriging.get_results()

JobPendingError: Job at https://350mt.api.integration.seequent.com/compute/orgs/829e6621-0ab6-4d7d-96bb-2bb5b407a5fe/geostatistics/kriging/5611c752-b7de-4851-890f-7279da07e462/status is still pending with status: in progress

## Retrieve the results
Get the updated masked grid 

In [24]:
# reload the modified block model with kriging results
mgrid = await object_client.download_object_by_path("INTR2-testevokriging.json")

Now, access the kriged estimates 

In [25]:
# retrieve the kriged values as a DataFrame
au_ok = await mgrid.download_dataframe(
    mgrid.search("cell_attributes[?name=='Au_ok2'].values")[0], fb=FeedbackWidget("Downloading DataFrame")
)

## Send results to Blocksync
Now get the blocksync model

In [26]:
# now need to access the blocksync model - now need to move the data from the `mgrid` geoscience objec to blocksync model
bms_client = BlockModelAPIClient(environment, connector, cache=manager.cache)

bm_uuid = "961a391f-d0d6-4627-80b2-eef62a324a93"  # had to query from portal url

Get the coordinates and categorical data to associate values with correct domain

In [27]:
# get the blockmodel coordinates and categories
bm = await bms_client.query_block_model_as_table(
    bm_id=bm_uuid, columns=["x", "y", "z", "GM"], column_headers=ColumnHeaderType.name
)
bm_df = bm.to_pandas()

Select a subset for only the domain of interest

In [28]:
# select only INTR2 blocks (this was the area where mask=True)
intr2_df = bm_df[bm_df["GM"] == "INTR2"].copy()

Sort the coordinates such that the are correctly associated with the values

In [29]:
# need to attach the kriged values to the intr2_df BUT the sort order isn't right.
# one way to handle this is to sort the coordinates before attaching to the kriged values
# but I havent been able to get it right :(
intr2_df.sort_values(by=["z", "x", "y"], inplace=True)

Give the column a name

In [30]:
# give the new column a name - annoyingly cant update the same column so each time I createa new one (hence the incrementing numbers)
col_name = "Au_ok-test"

Add the values to the dataframe with sorted coordinates

In [31]:
# attach grade values to the [sorted] intr2_df
intr2_df[col_name] = au_ok.values

Convert from pandas to pyarrow table to satisfy Blocksync

In [32]:
# convert to a pyarrow table for upload to blocksync
intr2_table = pa.Table.from_pandas(intr2_df[["x", "y", "z", col_name]], preserve_index=False)
intr2_table

pyarrow.Table
x: double
y: double
z: double
Au_ok-test: double
----
x: [[514402.5,514407.5,514407.5,514407.5,514407.5,...,514652.5,514657.5,514657.5,514657.5,514662.5]]
y: [[5686667.5,5686647.5,5686652.5,5686657.5,5686662.5,...,5686542.5,5686532.5,5686537.5,5686542.5,5686537.5]]
z: [[2.5,2.5,2.5,2.5,2.5,...,552.5,552.5,552.5,552.5,552.5]]
Au_ok-test: [[null,0.08,null,null,2.0131491719634025,...,0.2841685171475791,0.29476025092123337,0.3811576649869991,0.1220453466964816,0.3273103257207231]]

Upload to Blocksync

In [33]:
# upload to blocksync
version_with_new_columns = await bms_client.add_new_columns(
    bm_id=bm_uuid,
    data=intr2_table,
)

In [34]:
version_with_new_columns

Version(bm_uuid=UUID('961a391f-d0d6-4627-80b2-eef62a324a93'), version_id=24, version_uuid=UUID('05670711-f520-45db-a5b1-a3a3f18ad9d6'), parent_version_id=23, base_version_id=23, geoscience_version_id='1763763829789537964', created_at=datetime.datetime(2025, 11, 21, 22, 23, 49, 97213, tzinfo=TzInfo(0)), created_by=ServiceUser(id=UUID('5bda04fd-9826-46e8-b580-433578774853'), name='Eric Daniels', email='Eric.Daniels@bentley.com'), comment='', bbox=BBox(i_minmax=IntRange(max=259, min=0), j_minmax=IntRange(max=191, min=0), k_minmax=IntRange(max=131, min=0)), columns=[Column(col_id='i', data_type=<DataType.UInt32: 'UInt32'>, title='i', unit_id=None), Column(col_id='j', data_type=<DataType.UInt32: 'UInt32'>, title='j', unit_id=None), Column(col_id='k', data_type=<DataType.UInt32: 'UInt32'>, title='k', unit_id=None), Column(col_id='version_id', data_type=<DataType.UInt32: 'UInt32'>, title='version_id', unit_id=None), Column(col_id='sub_block_derivation', data_type=<DataType.Utf8: 'Utf8'>, titl